# EDA

In [6]:
import pandas as pd

df= pd.read_csv("../../data/raw/jobs.csv")
print(df.head())

   index          Job Title               Salary Estimate  \
0      0  Sr Data Scientist  $137K-$171K (Glassdoor est.)   
1      1     Data Scientist  $137K-$171K (Glassdoor est.)   
2      2     Data Scientist  $137K-$171K (Glassdoor est.)   
3      3     Data Scientist  $137K-$171K (Glassdoor est.)   
4      4     Data Scientist  $137K-$171K (Glassdoor est.)   

                                     Job Description  Rating  \
0  Description\n\nThe Senior Data Scientist is re...     3.1   
1  Secure our Nation, Ignite your Future\n\nJoin ...     4.2   
2  Overview\n\n\nAnalysis Group is one of the lar...     3.8   
3  JOB DESCRIPTION:\n\nDo you have a passion for ...     3.5   
4  Data Scientist\nAffinity Solutions / Marketing...     2.9   

              Company Name       Location            Headquarters  \
0         Healthfirst\n3.1   New York, NY            New York, NY   
1             ManTech\n4.2  Chantilly, VA             Herndon, VA   
2      Analysis Group\n3.8     Boston, MA

### Health check 

In [8]:
print("SHAPE:", df.shape)
print("\nCOLUMNS:", df.columns.tolist())
print("\nDTYPES:\n", df.dtypes)
print("\nSTATS:\n", df.describe(include="all"))

SHAPE: (672, 15)

COLUMNS: ['index', 'Job Title', 'Salary Estimate', 'Job Description', 'Rating', 'Company Name', 'Location', 'Headquarters', 'Size', 'Founded', 'Type of ownership', 'Industry', 'Sector', 'Revenue', 'Competitors']

DTYPES:
 index                  int64
Job Title                str
Salary Estimate          str
Job Description          str
Rating               float64
Company Name             str
Location                 str
Headquarters             str
Size                     str
Founded                int64
Type of ownership        str
Industry                 str
Sector                   str
Revenue                  str
Competitors              str
dtype: object

STATS:
              index       Job Title              Salary Estimate  \
count   672.000000             672                          672   
unique         NaN             172                           30   
top            NaN  Data Scientist  $75K-$131K (Glassdoor est.)   
freq           NaN             337

### check for nulls and duplicated values

In [9]:
print("\nNULLS:\n", df.isnull().sum())
print("\nDUPLICATES:", df.duplicated().sum())


NULLS:
 index                0
Job Title            0
Salary Estimate      0
Job Description      0
Rating               0
Company Name         0
Location             0
Headquarters         0
Size                 0
Founded              0
Type of ownership    0
Industry             0
Sector               0
Revenue              0
Competitors          0
dtype: int64

DUPLICATES: 0


### check logic

In [10]:
def check_logic(df: pd.DataFrame):
    print("--- Logical Consistency Check ---")
    # Check for the common Glassdoor '-1' placeholder
    minus_ones = (df == -1).sum().sum() + (df == "-1").sum().sum()
    print(f"Total '-1' placeholders found: {minus_ones}")
    
    # Check for impossible ratings
    out_of_bounds_rating = df[(df['Rating'] < 0) | (df['Rating'] > 5)].shape[0]
    print(f"Ratings outside 0-5 range: {out_of_bounds_rating}")
    
    # Check for unrealistic years
    current_year = 2026
    future_founded = df[df['Founded'] > current_year].shape[0]
    print(f"Companies founded in the future: {future_founded}")
check_logic(df)

--- Logical Consistency Check ---
Total '-1' placeholders found: 923
Ratings outside 0-5 range: 50
Companies founded in the future: 0


In [13]:
# Convert everything to string for a unified check (handles both -1 and "-1")
placeholder_counts = (df.astype(str) == '-1').sum()

# Filter to show only columns that actually have placeholders
report = placeholder_counts[placeholder_counts > 0].sort_values(ascending=False)

print("--- Columns containing '-1' placeholders ---")
if not report.empty:
    for col, count in report.items():
        percentage = (count / len(df)) * 100
        print(f"{col:20} | Found: {count:4} | ({percentage:.1f}%)")
else:
    print("✅ No '-1' placeholders found in any column!")

--- Columns containing '-1' placeholders ---
Competitors          | Found:  501 | (74.6%)
Founded              | Found:  118 | (17.6%)
Industry             | Found:   71 | (10.6%)
Sector               | Found:   71 | (10.6%)
Headquarters         | Found:   31 | (4.6%)
Size                 | Found:   27 | (4.0%)
Type of ownership    | Found:   27 | (4.0%)
Revenue              | Found:   27 | (4.0%)


In [14]:
# Filter rows where Founded is -1
founded_neg = df[df['Founded'] == -1]

print(f"--- Rows with negative 'Founded' values ---")
print(f"Count: {len(founded_neg)}")
print("\nSample of affected rows (Company Name and Founded):")
print(founded_neg[['Company Name', 'Founded', 'Size', 'Type of ownership']].head(10))

--- Rows with negative 'Founded' values ---
Count: 118

Sample of affected rows (Company Name and Founded):
                 Company Name  Founded                     Size  \
69                CareDx\n2.5       -1        1 to 50 employees   
112   Maxar Technologies\n3.5       -1  5001 to 10000 employees   
154  Covid-19 Search Partners       -1                       -1   
158       Radical Convergence       -1                       -1   
162   Maxar Technologies\n3.5       -1  5001 to 10000 employees   
193           SkillSoniq\n5.0       -1                  Unknown   
195        Joby Aviation\n4.3       -1      51 to 200 employees   
211   Maxar Technologies\n3.5       -1  5001 to 10000 employees   
230              Encode, Inc.       -1        1 to 50 employees   
236        Surya Systems\n4.6       -1        1 to 50 employees   

     Type of ownership  
69   Company - Private  
112   Company - Public  
154                 -1  
158                 -1  
162   Company - Public  
193 

In [16]:
# Count how many rows have -1 in the 'Founded' column
founded_minus_one_count = (df['Founded'] == -1).sum()

print(f"Number of companies with missing 'Founded' year (-1): {founded_minus_one_count}")

Number of companies with missing 'Founded' year (-1): 118
